# Neural Networks from Scratch — 4. MNIST and a Small CNN

**Notebook 4 of 4**

| # | Notebook | What you build |
|---|---|---|
| 1 | NumPy foundations | A single neuron, trained by hand |
| 2 | A network from scratch | A 2-layer network with backpropagation |
| 3 | The same network in PyTorch | Tensors, autograd, `nn.Module` |
| **4** | **MNIST and a small CNN** | **A real image classifier** |

---

## What this notebook is for

Everything so far has used 400 points on a plane. Now we use a real dataset:
**70,000 handwritten digits**, and a model that reads them at better than human
speed.

Three genuinely new ideas appear here, and they're the ones that separate a toy from
a working model:

1. **Mini-batches** — we stop feeding the whole dataset in at once
2. **The train/test split** — we finally stop marking our own homework, and
   overfitting becomes visible
3. **Convolutions** — layers built for the fact that images have *spatial structure*

The training loop itself does not change. It's still the five lines from notebook 3,
which are still the five steps from notebook 1.

## Running time

Everything here is CPU-only, but this notebook is slower than the others: the
fully-connected model takes a minute or two, and the CNN a few minutes. Cells that
take a while say so in advance and print progress as they go.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

torch.manual_seed(0)

plt.rcParams["axes.grid"] = False
plt.rcParams["figure.dpi"] = 110

print("torch      ", torch.__version__)
import torchvision
print("torchvision", torchvision.__version__)
print("threads    ", torch.get_num_threads())

---
# 6.1 The data

MNIST is 70,000 greyscale images of handwritten digits, 28×28 pixels each, labelled 0
to 9. It has been the "hello world" of computer vision since the 1990s.

Crucially, it arrives **already split** into 60,000 training images and 10,000 test
images. That split is not a formality — it's the entire basis for knowing whether a
model has learned anything.

## Transforms

`transforms.Compose` chains together the preprocessing applied to each image:

- **`ToTensor()`** converts the image to a tensor and rescales pixel values from
  0–255 into 0.0–1.0.
- **`Normalize(mean, std)`** subtracts the dataset mean and divides by its standard
  deviation, so pixel values end up centred around zero.

The magic numbers `0.1307` and `0.3081` are MNIST's actual mean and standard
deviation — everyone uses them and they're not something you'd derive. Normalising
matters because gradient descent behaves much better when inputs are centred and
similarly scaled; un-normalised inputs make the loss surface stretched and awkward to
descend.

The first run downloads about 10 MB, so you need an internet connection once. After
that it reads from the `data/` folder.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_data = datasets.MNIST(root="data", train=True, download=True, transform=transform)
test_data = datasets.MNIST(root="data", train=False, download=True, transform=transform)

print("training images:", len(train_data))
print("test images:    ", len(test_data))

image, label = train_data[0]
print()
print("one image:", tuple(image.shape), "-> (channels, height, width)")
print("its label:", label)
print(f"pixel range after normalisation: {image.min():.2f} to {image.max():.2f}")

> **If the download fails** (a corporate proxy, no internet, an SSL error), the
> dataset is small enough to fetch by other means. A working fallback is
> `sklearn.datasets.fetch_openml("mnist_784", version=1)`, which returns the same
> data as NumPy arrays you can wrap in a `TensorDataset`. The rest of the notebook
> only needs `train_data` and `test_data` to exist.

Note the shape: `(1, 28, 28)`. The leading `1` is the **channel** dimension — one
channel because the images are greyscale. A colour photo would be `(3, H, W)` for
red, green and blue. PyTorch always puts channels first for image data, and the CNN
later depends on that layout.

## Looking at the data

Always look at your data before modelling it. It takes two minutes and catches
mislabelled files, wrong scaling, and unpleasant surprises about what the images
actually contain.

In [ ]:
# A second copy without normalisation, purely so the images display with
# their natural black-and-white contrast.
display_data = datasets.MNIST(root="data", train=True, download=True,
                              transform=transforms.ToTensor())

fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))
for ax, idx in zip(axes.ravel(), range(16)):
    img, lab = display_data[idx]
    ax.imshow(img.squeeze(), cmap="gray")     # squeeze: (1,28,28) -> (28,28)
    ax.set_title(str(lab), fontsize=11)
    ax.axis("off")

plt.suptitle("MNIST: the first 16 training images", y=1.04)
plt.tight_layout()
plt.show()

In [ ]:
# An image IS just a grid of numbers. Here is a digit as its pixel values,
# rounded and with zeros blanked out so the shape shows through.
img, lab = display_data[0]
grid = (img.squeeze().numpy() * 9).round().astype(int)

print(f"label: {lab}\n")
for row in grid:
    print("".join(f"{v}" if v > 0 else "." for v in row))

That is the entire input to the model: 784 numbers. It has no notion of "a loop with
a tail" or even of which pixels are next to each other — at least, not until we get
to convolutions.

In [ ]:
# Check the classes are reasonably balanced. An unbalanced dataset needs
# different handling, so it is worth 30 seconds to look.
labels = train_data.targets.numpy()
counts = np.bincount(labels)

plt.figure(figsize=(6, 3))
plt.bar(range(10), counts)
plt.xticks(range(10))
plt.xlabel("digit"); plt.ylabel("count")
plt.title("Roughly balanced - no class needs special treatment")
plt.show()

print("smallest class:", counts.min(), " largest:", counts.max())

---
# 6.2 Mini-batches and `DataLoader`

In notebooks 1–3 every training step used the *entire* dataset: 400 points, one
matrix multiply, one gradient. That's called **full-batch** gradient descent, and
with 60,000 images it is a bad idea:

- 60,000 × 784 floats at once is a lot of memory, and a real dataset would be far
  worse
- you get exactly **one** parameter update per pass over the data, which is
  agonisingly slow progress

So instead we use **mini-batches**: take 64 images, compute the gradient from just
those, update, take the next 64. One pass through the whole dataset is an **epoch**,
and in our case it gives about 938 updates instead of 1.

The gradient from 64 images is a noisier estimate than the gradient from all 60,000 —
but it's roughly right, and you get it almost a thousand times more often. That trade
is overwhelmingly worth it. (The noise even helps: it can shake the model out of poor
local minima, like the ones that trapped the narrow networks in notebook 2.)

This is **stochastic gradient descent**, and it's what the "SGD" in `optim.SGD` has
meant all along.

## `DataLoader`

`DataLoader` wraps a dataset and hands out batches. It also **shuffles** between
epochs, which matters: if the data arrived sorted by digit, the model would see
nothing but 0s, then nothing but 1s, and would lurch around chasing whichever digit
it saw last.

In [ ]:
BATCH_SIZE = 64

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_data, batch_size=512, shuffle=False)
# Notes:
#  - shuffle=True for training, False for testing (order is irrelevant there,
#    and a bigger batch just makes evaluation faster).
#  - DataLoader takes a num_workers argument for parallel loading. Leave it at
#    the default of 0 inside notebooks, especially on Windows, where extra
#    worker processes tend to cause trouble.

print("batches per training epoch:", len(train_loader))

images, labels_batch = next(iter(train_loader))     # peek at one batch
print()
print("images shape:", tuple(images.shape), "-> (batch, channels, height, width)")
print("labels shape:", tuple(labels_batch.shape), "dtype", labels_batch.dtype)
print("first 10 labels:", labels_batch[:10].tolist())

`(64, 1, 28, 28)` is the shape you'll be reading for the rest of your deep learning
life: **batch, channels, height, width**. Say it out loud once; it saves a lot of
confusion later.

---
# 6.3 A fully connected network

Start with what you already know: the notebook 3 network, made bigger and with ten
outputs instead of one.

Two changes are needed for ten classes.

**`nn.Flatten()`** turns each `(1, 28, 28)` image into a flat row of 784 numbers,
because `nn.Linear` expects rows of features. Note what this throws away: after
flattening, the model has no idea that pixel 0 and pixel 1 were neighbours. It has to
*learn* that from scratch. Holding on to that information is exactly what
convolutions will do for us later.

**`nn.CrossEntropyLoss`** is the multi-class version of binary cross-entropy. It
wants:

- the model's raw scores (**logits**), shape `(batch, 10)` — one score per digit
- the correct labels as integers, shape `(batch,)` — not one-hot vectors

Like `BCEWithLogitsLoss` from notebook 3, it applies the softmax internally for
numerical stability. **So the model must not end with a softmax layer.** Adding one is
the same double-application bug, and it trains quietly and badly.

In [ ]:
class MLP(nn.Module):
    """A plain fully connected network: 784 -> 128 -> 10."""

    def __init__(self, n_hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),                  # (B, 1, 28, 28) -> (B, 784)
            nn.Linear(28 * 28, n_hidden),
            nn.ReLU(),
            nn.Linear(n_hidden, 10),       # 10 logits, NO softmax
        )

    def forward(self, x):
        return self.net(x)


torch.manual_seed(0)
mlp = MLP()
print(mlp)
print()
print("parameters:", sum(p.numel() for p in mlp.parameters()))

About 100,000 parameters, nearly all of them in that first `784 -> 128` layer.

Now the training and evaluation functions. Read them closely: apart from the loop
over batches, `train_one_epoch` is the same five lines from notebook 3.

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer):
    """One pass over the training data. Returns average loss and accuracy."""
    model.train()                      # training mode
    total_loss, n_correct, n_seen = 0.0, 0, 0

    for images, labels in loader:
        optimizer.zero_grad()              # 1. clear gradients
        logits = model(images)             # 2. forward
        loss = loss_fn(logits, labels)     # 3. loss
        loss.backward()                    # 4. backward
        optimizer.step()                   # 5. update

        total_loss += loss.item() * labels.size(0)
        n_correct += (logits.argmax(dim=1) == labels).sum().item()
        n_seen += labels.size(0)

    return total_loss / n_seen, n_correct / n_seen


@torch.no_grad()                       # nothing in here needs gradients
def evaluate(model, loader, loss_fn):
    """Measure loss and accuracy without training. Returns (loss, accuracy)."""
    model.eval()                       # evaluation mode
    total_loss, n_correct, n_seen = 0.0, 0, 0

    for images, labels in loader:
        logits = model(images)
        total_loss += loss_fn(logits, labels).item() * labels.size(0)
        n_correct += (logits.argmax(dim=1) == labels).sum().item()
        n_seen += labels.size(0)

    return total_loss / n_seen, n_correct / n_seen

Three details in there that are easy to skip over and important to get right:

- **`logits.argmax(dim=1)`** turns ten scores into one prediction — the index of the
  biggest. `dim=1` means "across the 10 classes", not across the batch.
- **`model.train()` / `model.eval()`** switch behaviour for layers like dropout and
  batch-norm, which act differently during training and prediction. Our models don't
  contain any, so it changes nothing today — but forgetting it with a model that does
  is a genuinely baffling bug, and the habit costs nothing.
- **`@torch.no_grad()`** as a decorator turns off gradient tracking for the whole
  function: faster, and less memory. Use it for anything that isn't training.

In [ ]:
# About 30-60 seconds per epoch on a typical laptop CPU.
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(mlp.parameters(), lr=1e-3)

N_EPOCHS = 3
history = {"train_acc": [], "test_acc": [], "train_loss": [], "test_loss": []}

for epoch in range(1, N_EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(mlp, train_loader, loss_fn, optimizer)
    te_loss, te_acc = evaluate(mlp, test_loader, loss_fn)

    history["train_loss"].append(tr_loss); history["train_acc"].append(tr_acc)
    history["test_loss"].append(te_loss); history["test_acc"].append(te_acc)

    print(f"epoch {epoch}  "
          f"train loss {tr_loss:.4f} acc {tr_acc:.2%}   |   "
          f"test loss {te_loss:.4f} acc {te_acc:.2%}   "
          f"({time.time() - t0:.0f}s)")

Around 97% on digits the model has never seen — about one mistake in every 33 images,
from a model with no idea what a digit is, trained in a couple of minutes on a
laptop.

**The number that matters is the test accuracy**, because it's the only one measured
on data the model didn't learn from. From here on we always report both, and the gap
between them is the thing to watch.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
epochs = range(1, N_EPOCHS + 1)

axes[0].plot(epochs, history["train_loss"], "o-", label="train")
axes[0].plot(epochs, history["test_loss"], "o-", label="test")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].legend()
axes[0].set_title("Loss"); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, history["train_acc"], "o-", label="train")
axes[1].plot(epochs, history["test_acc"], "o-", label="test")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy"); axes[1].legend()
axes[1].set_title("Accuracy"); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
# 6.4 Looking at the mistakes

An accuracy number tells you *how much* the model gets wrong. Looking at the errors
tells you *what kind* of wrong, which is what actually guides your next move.

In [ ]:
@torch.no_grad()
def collect_predictions(model, loader):
    """Run the whole test set and return predictions, labels, and images."""
    model.eval()
    preds, labs, imgs = [], [], []
    for images, labels in loader:
        preds.append(model(images).argmax(dim=1))
        labs.append(labels)
        imgs.append(images)
    return torch.cat(preds), torch.cat(labs), torch.cat(imgs)


pred, true, imgs = collect_predictions(mlp, test_loader)
wrong = (pred != true).nonzero(as_tuple=True)[0]      # indices of the mistakes

print(f"{len(wrong)} mistakes out of {len(true)}  ({len(wrong) / len(true):.2%} error rate)")

In [ ]:
# Un-normalise for display: we subtracted the mean and divided by the std,
# so multiply and add back.
def unnormalise(t):
    return t * 0.3081 + 0.1307


fig, axes = plt.subplots(2, 8, figsize=(12, 3.6))
for ax, idx in zip(axes.ravel(), wrong[:16]):
    ax.imshow(unnormalise(imgs[idx]).squeeze(), cmap="gray")
    ax.set_title(f"said {pred[idx].item()}, was {true[idx].item()}", fontsize=8)
    ax.axis("off")

plt.suptitle("Where it went wrong", y=1.04)
plt.tight_layout()
plt.show()

Look at these properly. Most of them are genuinely ambiguous — sloppy 4s that look
like 9s, 5s that could be 3s, digits a human would also hesitate over. That's a good
sign: the model is failing on hard cases, not on easy ones. If it were getting clean,
obvious digits wrong, something would be structurally broken.

## The confusion matrix

Which digits get mistaken for which? Row = true label, column = what the model said.
The diagonal is correct; everything off it is an error.

In [ ]:
cm = np.zeros((10, 10), dtype=int)
for t, p in zip(true.numpy(), pred.numpy()):
    cm[t, p] += 1

fig, ax = plt.subplots(figsize=(6.5, 5.5))
# Hide the diagonal so the colour scale is set by the errors, not by the
# (much larger) correct counts.
off_diagonal = cm.copy()
np.fill_diagonal(off_diagonal, 0)
im = ax.imshow(off_diagonal, cmap="Reds")

for i in range(10):
    for j in range(10):
        if i != j and cm[i, j] > 0:
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8)

ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xlabel("predicted"); ax.set_ylabel("true label")
ax.set_title("Confusion matrix (diagonal removed)")
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

worst = np.dstack(np.unravel_index(np.argsort(off_diagonal.ravel())[::-1][:5],
                                   (10, 10)))[0]
print("most common confusions:")
for t, p in worst:
    print(f"  true {t} predicted as {p}: {cm[t, p]} times")

The confusions are the ones you'd expect from the shapes: 4 and 9, 7 and 2, 3 and 5,
8 and 5. The model has learned something about how digits look, not a lookup table.

This kind of plot is how you decide what to do next in a real project — whether you
need more data of a particular class, a better architecture, or simply cleaner labels.

---
# 6.5 🔬 Overfitting, made visible

In notebook 2 we suspected the 128-unit network was starting to memorise noise, but
couldn't prove it, because we only ever scored it on its own training data. Now we
have a test set, so let's show overfitting properly.

The recipe: take a **big** model and a **tiny** dataset. 500 training images, a model
with far more parameters than that, and plenty of epochs.

In [ ]:
# 60 epochs, but on only 500 images - the time goes on evaluating the full
# test set each epoch. Expect a couple of minutes.
torch.manual_seed(0)

small_train = Subset(train_data, range(500))          # just 500 images
small_loader = DataLoader(small_train, batch_size=32, shuffle=True)

big_model = MLP(n_hidden=256)
print("parameters:", sum(p.numel() for p in big_model.parameters()),
      "  training examples:", len(small_train))
print("-> roughly 400 parameters per training example. Ample room to memorise.")

opt = optim.Adam(big_model.parameters(), lr=1e-3)
fn = nn.CrossEntropyLoss()

hist = {"train": [], "test": []}
for epoch in range(60):
    _, tr_acc = train_one_epoch(big_model, small_loader, fn, opt)
    _, te_acc = evaluate(big_model, test_loader, fn)
    hist["train"].append(tr_acc)
    hist["test"].append(te_acc)
    if epoch % 10 == 0 or epoch == 59:
        print(f"epoch {epoch:3}  train {tr_acc:.2%}   test {te_acc:.2%}   "
              f"gap {tr_acc - te_acc:+.1%}")

In [ ]:
plt.figure(figsize=(7.5, 4.5))
plt.plot(hist["train"], label="training accuracy", linewidth=2)
plt.plot(hist["test"], label="test accuracy", linewidth=2)
plt.fill_between(range(len(hist["train"])), hist["train"], hist["test"],
                 alpha=0.15, color="red")
plt.xlabel("epoch"); plt.ylabel("accuracy")
plt.title("Overfitting: the shaded gap is memorisation, not learning")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

There it is. **Training accuracy climbs to 100% and stays there. Test accuracy
plateaus far below and stops improving.** The red gap is the model memorising its 500
examples rather than learning what digits look like.

Two lessons, and they are among the most important in the whole course:

**1. Training accuracy is not a measure of anything useful.** A model can hit 100% on
training data and be worthless. If someone quotes you an accuracy figure, the first
question is always "on what data?"

**2. The remedy is usually more data, not a better model.** Compare this run to the
first one: the same architecture on all 60,000 images reached ~97% with barely any
gap. Nothing about the model changed. It just had enough examples that memorising
them individually stopped being the easy option.

When more data isn't available, the toolbox is **regularisation** — dropout, weight
decay, data augmentation, early stopping. All are a few lines in PyTorch, and all are
worth reading about once this notebook makes sense.

**🔬 Try it.** Change `range(500)` to `range(5000)` and re-run. Watch the gap shrink.

---
# 7.1 Why images need convolutions

Our fully connected model flattens the image into 784 unrelated numbers. Two problems
follow, and both are fundamental:

**It doesn't know that nearby pixels are related.** Pixel 100 and pixel 101 are
neighbours in the image, but to `nn.Linear` they are just columns 100 and 101 — no
more related than any other pair. The model has to learn the entire 2D structure of
an image from scratch, from examples.

**It learns every position separately.** If it learns to spot a horizontal stroke in
the top-left corner, that knowledge does nothing for a horizontal stroke in the
middle. Each position is learned independently, which wastes both parameters and
training data.

A **convolutional layer** fixes both. It slides a small window — typically 3×3 —
across the image, applying the *same* small set of weights at every position. That
window is called a **filter** or **kernel**.

- because it's small and local, it naturally works on neighbouring pixels
- because the same weights are used everywhere, a pattern learned in one place is
  detected everywhere. This is **weight sharing**, and it's why CNNs need far fewer
  parameters than a dense layer doing the same job.

## Seeing it work

Convolution isn't a neural network idea — it's ordinary image processing, and it's
what image filters in Photoshop do. Here are three hand-written kernels applied to a
digit, so you can see what a filter *is* before we ask a network to learn some.

In [ ]:
import torch.nn.functional as F

img, lab = display_data[0]
x = img.unsqueeze(0)                       # (1,28,28) -> (1,1,28,28): conv2d wants a batch

kernels = {
    "original": None,
    "vertical edges": [[-1, 0, 1],
                       [-2, 0, 2],
                       [-1, 0, 1]],
    "horizontal edges": [[-1, -2, -1],
                         [0, 0, 0],
                         [1, 2, 1]],
    "blur": [[1 / 9] * 3] * 3,
}

fig, axes = plt.subplots(1, 4, figsize=(11, 3))
for ax, (name, k) in zip(axes, kernels.items()):
    if k is None:
        out = img.squeeze()
    else:
        kernel = torch.tensor(k, dtype=torch.float32).reshape(1, 1, 3, 3)
        out = F.conv2d(x, kernel, padding=1).squeeze()
    ax.imshow(out.detach(), cmap="gray")
    ax.set_title(name, fontsize=10)
    ax.axis("off")

plt.suptitle(f"The same 3x3 kernel applied at every position (digit: {lab})", y=1.05)
plt.tight_layout()
plt.show()

Nine numbers, arranged three different ways, and you get an edge detector, a different
edge detector, and a blur. A convolutional layer contains many such kernels —
**and it learns their numbers by gradient descent, exactly like any other weights.**

That's the whole idea. The first layer typically learns edge and blob detectors much
like these; later layers combine those into corners, curves and strokes; later still
into whole digits. Nobody designs that hierarchy — it falls out of training.

## The supporting cast

Three more pieces and we can build the model.

**`nn.Conv2d(in_channels, out_channels, kernel_size, padding)`** — the layer. It
learns `out_channels` separate kernels, each looking at all `in_channels` of its
input. `padding=1` with a 3×3 kernel keeps the output the same size as the input
(without it, the image shrinks by 2 pixels each time).

**`nn.MaxPool2d(2)`** — takes the biggest value in each 2×2 block, halving the height
and width. It throws away exact positions while keeping "was this pattern present
nearby?", which makes the model more robust to small shifts and cuts the work for the
layers above.

**Channels.** A colour image has 3 channels; MNIST has 1. After a conv layer with 16
kernels, it has 16 — one "feature map" per kernel, each one a picture of where that
kernel found its pattern. So channels start as colours and become *learned features*.

Here's the shape journey through the model we're about to build:

```
input                        (B,  1, 28, 28)
Conv2d(1 -> 16) + ReLU       (B, 16, 28, 28)    padding keeps the size
MaxPool2d(2)                 (B, 16, 14, 14)    halved
Conv2d(16 -> 32) + ReLU      (B, 32, 14, 14)
MaxPool2d(2)                 (B, 32,  7,  7)    halved again
Flatten                      (B, 1568)          32 * 7 * 7 = 1568
Linear(1568 -> 10)           (B, 10)            the ten logits
```

Trace those shapes yourself. Getting the flatten size right (1568) is the classic
beginner stumble, and the arithmetic above is how you work it out.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),    # (B, 16, 28, 28)
            nn.ReLU(),
            nn.MaxPool2d(2),                               # (B, 16, 14, 14)
            nn.Conv2d(16, 32, kernel_size=3, padding=1),   # (B, 32, 14, 14)
            nn.ReLU(),
            nn.MaxPool2d(2),                               # (B, 32,  7,  7)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),                                  # (B, 1568)
            nn.Linear(32 * 7 * 7, 10),                     # (B, 10)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


torch.manual_seed(0)
cnn = SmallCNN()

# Confirm the shapes before training anything. Ten seconds here saves ten
# minutes of confusing errors later - and it is what the pros do.
dummy = torch.zeros(4, 1, 28, 28)         # a fake batch of 4 images
print("input           ", tuple(dummy.shape))
h = dummy
for layer in cnn.features:
    h = layer(h)
    print(f"{layer.__class__.__name__:12}    {tuple(h.shape)}")
print("after classifier", tuple(cnn.classifier(h).shape))

print()
print("CNN parameters:", sum(p.numel() for p in cnn.parameters()))
print("MLP parameters:", sum(p.numel() for p in mlp.parameters()))

Note the parameter counts: the CNN has **fewer** parameters than the fully connected
model, and it is about to do considerably better. That's weight sharing paying off —
the same kernel reused at all 784 positions instead of separate weights for each.

More capacity is not what makes a model better. Matching the model's structure to the
structure of the data is.

In [ ]:
# Slower than the MLP: expect a few minutes per epoch on a laptop CPU.
# If that's too slow, drop to N_EPOCHS_CNN = 1 - one epoch already beats
# the fully connected model.
torch.manual_seed(0)
cnn = SmallCNN()

cnn_loader = DataLoader(train_data, batch_size=128, shuffle=True)   # bigger batch, faster
optimizer = optim.Adam(cnn.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

N_EPOCHS_CNN = 2
cnn_hist = {"train_acc": [], "test_acc": []}

for epoch in range(1, N_EPOCHS_CNN + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(cnn, cnn_loader, loss_fn, optimizer)
    te_loss, te_acc = evaluate(cnn, test_loader, loss_fn)
    cnn_hist["train_acc"].append(tr_acc)
    cnn_hist["test_acc"].append(te_acc)
    print(f"epoch {epoch}  train acc {tr_acc:.2%}   test acc {te_acc:.2%}   "
          f"({time.time() - t0:.0f}s)")

In [ ]:
mlp_best = max(history["test_acc"])
cnn_best = max(cnn_hist["test_acc"])

print(f"fully connected : {mlp_best:.2%} test accuracy")
print(f"convolutional   : {cnn_best:.2%} test accuracy")
print()
err_mlp, err_cnn = 1 - mlp_best, 1 - cnn_best
print(f"error rate {err_mlp:.2%} -> {err_cnn:.2%}, "
      f"a {(err_mlp - err_cnn) / err_mlp:.0%} reduction in mistakes")

The accuracy difference looks small — a point or so. Look at it as the **error rate**
instead: roughly a third to a half of the remaining mistakes, gone.

That reframing is a useful habit. Going from 97% to 99% is not "2% better"; it is
"three times fewer errors", and at 99.9% each further step gets exponentially harder.
Late-stage accuracy improvements are always worth more than they look.

## What the first layer learned

In [ ]:
# The 16 kernels of the first conv layer, as 3x3 images.
filters = cnn.features[0].weight.detach()        # (16, 1, 3, 3)
print("first conv layer weights:", tuple(filters.shape))

fig, axes = plt.subplots(2, 8, figsize=(10, 2.8))
for ax, f in zip(axes.ravel(), filters):
    ax.imshow(f.squeeze(), cmap="RdBu")
    ax.axis("off")
plt.suptitle("The 16 learned 3x3 kernels (red positive, blue negative)", y=1.06)
plt.tight_layout()
plt.show()

In [ ]:
# And what one image looks like after passing through them.
img, lab = test_data[0]
with torch.no_grad():
    maps = cnn.features[0](img.unsqueeze(0)).squeeze(0)     # (16, 28, 28)

fig, axes = plt.subplots(2, 8, figsize=(12, 3.2))
for ax, m in zip(axes.ravel(), maps):
    ax.imshow(m, cmap="viridis")
    ax.axis("off")
plt.suptitle(f"16 feature maps for a '{lab}' - each kernel highlights something different",
             y=1.06)
plt.tight_layout()
plt.show()

Compare these with the hand-made edge detectors from earlier. Some look similar —
directional edge detectors — and others are harder to interpret. The network was
never told to look for edges; it discovered that edges are useful, because gradient
descent found that they reduce the loss.

Scale this up and the same thing happens with photographs: the first layer of a large
vision model learns edges and colour blobs, and the layers above compose them into
textures, then object parts, then objects. The mechanism is the one you derived in
notebook 2.

---
# 7.2 Wrap-up

## What you built

Over four notebooks, starting from nothing but NumPy:

| Notebook | What you built | The idea it taught |
|---|---|---|
| 1 | A single neuron, trained by hand | Forward, loss, backward, update |
| 2 | A 2-layer network with backprop | Depth needs non-linearity; blame flows backwards |
| 3 | The same network in PyTorch | Autograd does exactly what you did, automatically |
| 4 | An MNIST classifier and a CNN | Batches, the test set, and structure matching the data |

And underneath all four, unchanged:

```python
optimizer.zero_grad()      # clear
output = model(batch)      # forward
loss = loss_fn(output, y)  # measure
loss.backward()            # blame
optimizer.step()           # nudge
```

Those five lines train a 2-neuron toy and a frontier language model alike. The
difference is what sits in `model`, how much data goes through it, and how many
machines it takes.

## The habits worth keeping

- **Print shapes.** Most bugs are shape bugs.
- **Look at your data** before modelling it, and **look at your errors** afterwards.
- **Never trust training accuracy.** Ask "on what data?" every time.
- **Check the loss curve.** Flat, smooth, jagged, exploding — each has a diagnosis.
- **Start small.** Get a tiny model working on a tiny dataset first, then scale up.
- **When training misbehaves, suspect the learning rate first.**

## Where to go next

**To deepen the fundamentals**

- Andrej Karpathy, *Neural Networks: Zero to Hero* (YouTube). Follows almost exactly
  the path of these notebooks, then keeps going all the way to a small GPT. The
  natural next step.
- *Dive into Deep Learning* (d2l.ai) — free, thorough, code alongside the maths.

**To build things**

- fast.ai's *Practical Deep Learning for Coders* — top-down, results first.
- The official PyTorch tutorials at pytorch.org/tutorials.
- Hugging Face for pretrained models. In practice you rarely train from scratch: you
  take a model someone else trained on a huge dataset and fine-tune it on yours.

**Concepts worth learning next, roughly in order**

1. **Regularisation** — dropout, weight decay, data augmentation, early stopping. The
   direct answer to the overfitting you saw in 6.5.
2. **Better training** — learning rate schedules, batch normalisation.
3. **Transfer learning** — the single highest-value technique for practical work.
4. **Architectures** — ResNets for images, then attention and transformers.

## A last experiment

Take the CNN you just trained, draw your own digit (in any paint program, 28×28,
white on black), load it, and see what the model says. It is a good reminder that a
model trained on one particular dataset can be oddly fragile on data that looks
slightly different — a phenomenon called **distribution shift**, and one of the main
reasons deployed models disappoint.

### Check yourself

1. Why do we use mini-batches instead of the whole dataset at once? Name two reasons.
2. What does `shuffle=True` protect against?
3. A model gets 100% training accuracy and 82% test accuracy. What is happening, and
   what is the most reliable fix?
4. Why does the CNN beat the fully connected model despite having fewer parameters?
5. Your model ends with `nn.Softmax(dim=1)` and you use `nn.CrossEntropyLoss()`. What
   goes wrong, and will you get an error message?
6. You change `MaxPool2d(2)` to `MaxPool2d(4)`. What number in the model must change
   too, and what is its new value?

### Stretch goals

- Add `nn.Dropout(0.25)` before the final linear layer and re-run the 6.5 overfitting
  experiment. Watch the gap shrink.
- Add a third conv block and see whether it helps (watch the shape arithmetic).
- Try Fashion-MNIST — `datasets.FashionMNIST`, identical shapes, much harder.
- Save your trained CNN with `torch.save(cnn.state_dict(), "cnn.pt")` and write a
  small function that loads it and classifies a single image.

---

That's the course. You started with `x @ w + b` and ended with a convolutional
network — and, more usefully, with a clear picture of what is happening inside every
line of it.